# Study 1 and 2

- Framing (Fulfilment vs. Justice) × Comparability (One Family vs. Two Families)
- 9 Endowment Cases (Study 1: randomized, Study 2: ascending order)
- Numerical evaluation (without restrictions)

**Group 1 (Fulfilment) and 2 (Justice)**

| Case       | Family A   |
|:----------:|:----------:|
| **Need**   | **80**     |
|  1         |    0       |
|  2         |   20       |
|  3         |   40       |
|  4         |   60       |
|  5         |   80       |
|  6         |  100       |
|  7         |  120       |
|  8         |  140       |
|  9         |  160       |

**Group 3 (Fulfilment) and 4 (Justice)**

| Case       | Family A   | Family B   | Deviation (A)   | Ratio (A)        |
|:----------:|:----------:|:----------:|:---------------:|:----------------:|
| **Need**   | **80**     | **120**    | **(B = 30)**    | **(B = 0,60)**   |
|  1         |    0       |    90      | 80              | 0,00             |
|  2         |   20       |    90      | 60              | 0,25             |
|  3         |   40       |    90      | 40              | 0,50             |
|  4         |   60       |    90      | 20              | 0,75             |
|  5         |   80       |    90      |  0              | 1,00             |
|  6         |  100       |    90      | 20              | 1,25             |
|  7         |  120       |    90      | 40              | 1,50             |
|  8         |  140       |    90      | 60              | 1,75             |
|  9         |  160       |    90      | 80              | 2,00             |

## Set Up Notebook

In [ ]:
# Import packages
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import ttest_ind

## Clean Data

In [ ]:
# Load data
df = pd.read_csv('need_fulfilment_data_1_3.csv')
# df = pd.read_csv('need_fulfilment_data_2.csv')

# Filter for groups
df = df[df['randnumber'].isin([1, 2, 3, 4])]

# Define case columns
case_cols = [f'case1a{i}' for i in range(1, 10)] + \
            [f'case1b{i}' for i in range(1, 10)] + \
            [f'case1c{i}' for i in range(1, 10)] + \
            [f'case1d{i}' for i in range(1, 10)]

# Count answers for Control Question 1
control_1_counts = df['kontrolle1'].value_counts().sort_index()

# Calculate percentages for Control Question 1
control_1_pct = df['kontrolle1'].value_counts(normalize = True).sort_index() * 100

# Display results
print("\nControl Question 1")
print("\nCounts\n", control_1_counts)
print("\nPercentages\n", control_1_pct)

# Count answers for Control Question 2
control_2_counts = df['kontrolle2b'].value_counts().sort_index()

# Calculate percentages for Control Question 2
control_2_pct = df['kontrolle2b'].value_counts(normalize = True).sort_index() * 100

# Display results
print("\nControl Question 2")
print("\nCounts\n", control_2_counts)
print("\nPercentages\n", control_2_pct)

# Check if a subject has variance in responses
numeric_cases = df[case_cols].apply(pd.to_numeric, errors = 'coerce')
has_variance = numeric_cases.nunique(axis = 1) > 1

# Create DataFrame
df_clean = df[
    (df['kontrolle1'] == 7) &
    (df['kontrolle2b'] == 1) &
    (df[case_cols].notna().any(axis = 1)) &
    has_variance
].copy()

# Count participants per group
group_counts = df_clean['randnumber'].value_counts().sort_index()

# Display results
print("\nParticipants per group")
print(group_counts)
print(f"\nTotal sample size\n{len(df_clean)}")

## Calculate Sociodemographics

In [ ]:
# Define labels
label_dicts = {
    'geschlecht': {
        1: 'Female',
        2: 'Non-Binary',
        3: 'Male'
    },
    'bildung': {
        1: 'Still in School',
        2: 'Lower Secondary Education',
        3: 'Intermediate Secondary Education',
        4: 'Higher Education Entrance Qualification',
        5: 'Completed Vocational Training',
        6: 'Master Craftsman',
        7: 'Bachelor',
        8: 'Master',
       11: 'Doctorate'
    },
    'bundesland': {
        1: 'Baden-Wuerttemberg',
        2: 'Bavaria',
        3: 'Berlin',
        4: 'Brandenburg',
        5: 'Bremen',
        6: 'Hamburg',
        7: 'Hesse',
        8: 'Mecklenburg-Western Pomerania',
        9: 'Lower Saxony',
       10: 'North Rhine-Westphalia',
       11: 'Rhineland-Palatinate',
       12: 'Saarland',
       13: 'Saxony',
       14: 'Saxony-Anhalt',
       15: 'Schleswig-Holstein',
       16: 'Thuringia'
    }
}

# Iterate through continuous variables
for col in ['alter', 'politik', 'wuerde', 'wohnraum', 'einkommen', 'mitbewohner']:
    
    # Ensure data is numeric
    numeric_data = pd.to_numeric(df_clean[col], errors = 'coerce')
    
    # Calculate metrics
    col_mean = numeric_data.mean()
    col_std  = numeric_data.std()
    col_min  = numeric_data.min()
    col_max  = numeric_data.max()
    
    # Display results
    print(f"\n{col}:")
    print(f"Mean: {col_mean:.2f} | Std: {col_std:.2f} | Range: {col_min} - {col_max}")

# Iterate through categorical variables
for col, mapping in label_dicts.items():
    
    # Map codes to labels
    mapped_data = df_clean[col].map(mapping)
    
    # Calculate metrics
    counts = mapped_data.value_counts()
    percentages = mapped_data.value_counts(normalize = True) * 100
    
    # Create DataFrame
    df_summary = pd.DataFrame({
        'Counts': counts,
        'Percentages': percentages
    }).round(3)
    
    # Display results
    print(f"\n{col}:")
    print(df_summary)

## Compare Groups

In [ ]:
# Define group settings
group_infos = {
    1: {'cols': [f'case1a{i}' for i in range(1, 10)], 'name': 'G1 (Fulfillment, One Family)',   'color': 'black', 'fmt': '-o'},
    2: {'cols': [f'case1b{i}' for i in range(1, 10)], 'name': 'G2 (Justice, One Family)',       'color': 'black', 'fmt': '--s'},
    3: {'cols': [f'case1c{i}' for i in range(1, 10)], 'name': 'G3 (Fulfillment, Two Families)', 'color': 'black', 'fmt': '-^'},
    4: {'cols': [f'case1d{i}' for i in range(1, 10)], 'name': 'G4 (Justice, Two Families)',     'color': 'black', 'fmt': '--d'}
}

# Define endowments
endowment = [0, 20, 40, 60, 80, 100, 120, 140, 160]

# Calculate means and confidence intervals
def get_stats_for_group(group_num):
    
    # Retrieve column names
    cols = group_infos[group_num]['cols']
    
    # Filter for group
    df_group = df_clean[df_clean['randnumber'] == group_num][cols].copy()
    
    # Iterate through cases
    for col in cols:
        
        # Ensure all columns are numeric
        df_group[col] = pd.to_numeric(df_group[col], errors = 'coerce')
    
    # Extract raw values
    raw_data = df_group.values
    
    # Calculate row-wise minimum and maximum
    with np.errstate(all = 'ignore'):
        row_min = np.nanmin(raw_data, axis = 1, keepdims = True)
        row_max = np.nanmax(raw_data, axis = 1, keepdims = True)
    
    # Calculate range and handle zero-variance cases to avoid division by zero
    diff = row_max - row_min
    diff[diff == 0] = np.nan
    
    # Apply subject-level normalization
    norm_data = 2 * (raw_data - row_min) / diff - 1
    
    # Create DataFrame
    df_norm = pd.DataFrame(norm_data, columns = cols)
    
    # Initialize result lists
    means, cis = [], []
    
    # Iterate through cases
    for col in cols:
        
        # Extract non-missing normalized values
        vals = df_norm[col].dropna().values
        
        # Count number of observations
        n = len(vals)
        
        # Calculate mean and confidence intervals
        if n > 1:
            means.append(np.mean(vals))
            sem = np.std(vals, ddof = 1) / np.sqrt(n)
            cis.append(1.96 * sem)
        else:
            means.append(np.nan)
            cis.append(np.nan)
    
    # Return means and confidence intervals
    return means, cis

# Initialize dictionary
stats = {}

# Iterate through groups
for g in [1, 2, 3, 4]:
    
    # Precompute and store results
    stats[g] = get_stats_for_group(g)

# Define comparisons
comparisons = [
    (1, 2),
    (3, 4),
    (1, 3),
    (2, 4)
]

# Iterate through comparisons
for gA, gB in comparisons:
    
    # Unpack precomputed means and confidence intervals
    meanA, ciA = stats[gA]
    meanB, ciB = stats[gB]
    
    # Retrieve display settings
    infoA = group_infos[gA]
    infoB = group_infos[gB]
    
    # Construct DataFrame
    df_comp = pd.DataFrame({
        'Endowment': endowment,
        f'{infoA["name"]} (Mean)': meanA,
        f'95% CI ({gA})': ciA,
        f'{infoB["name"]} (Mean)': meanB,
        f'95% CI ({gB})': ciB
    })
    
    # Print section header
    print(f"Group {gA} vs. Group {gB}")
    
    # Display results
    try:
        display(df_comp.round(3))
    except NameError:
        print(df_comp.round(3).to_string(index = False))
    
    # Initialize figure
    plt.figure(figsize = (10, 6))
    
    # Plot means with error bars for Group A
    plt.errorbar(endowment,
                 meanA,
                 yerr = ciA,
                 fmt = infoA['fmt'],
                 label = infoA['name'],
                 capsize = 5,
                 color = infoA['color']
                )
    
    # Plot means with error bars for Group B
    plt.errorbar(endowment,
                 meanB,
                 yerr = ciB,
                 fmt = infoB['fmt'],
                 label = infoB['name'],
                 capsize = 5,
                 color = infoB['color']
                )
    
    # Add reference lines for need (vertical) and neutral point (horizontal)
    plt.axvline(x = 80, color = 'gray',  linestyle = ':', label = 'Need Family A (80)')
    plt.axhline(y =  0, color = 'black', linewidth = 0.5)
    
    # Set title
    plt.title(f'Group {gA} vs. Group {gB}')
    
    # Set labels
    plt.xlabel('Endowment of Family A')
    plt.ylabel('Mean Rating, Normalized to [-1, 1]')
    
    # Add legend
    plt.legend()
    
    # Add grid
    plt.grid(True, alpha = 0.3)
    
    # Export graph
    plt.savefig(f'need_fulfilment_study_1_line_plot_{gA}_vs_{gB}.pdf', format = 'pdf', bbox_inches = 'tight')
    # plt.savefig(f'need_fulfilment_study_2_line_plot_{gA}_vs_{gB}.pdf', format = 'pdf', bbox_inches = 'tight')
    
    # Display graph
    plt.show()

## Create Individual Plots

In [ ]:
# Set rows
cols_per_row = 5

# Iterate through groups
for g in [1, 2, 3, 4]:
    
    # Retrieve group information
    info = group_infos[g]
    cols = info['cols']
    
    # Filter for group
    df_group = df_clean[df_clean['randnumber'] == g].copy()
    
    # Extract raw values
    raw_data = df_group[cols].apply(pd.to_numeric, errors='coerce').values
    
    # Calculate row-wise minimum and maximum
    row_min = np.nanmin(raw_data, axis = 1, keepdims = True)
    row_max = np.nanmax(raw_data, axis = 1, keepdims = True)

    # Calculate range and handle zero-variance cases to avoid division by zero
    diff = row_max - row_min
    diff[diff == 0] = np.nan

    # Apply subject-level normalization
    norm_data = 2 * (raw_data - row_min) / diff - 1
    
    # Calculate grid dimensions
    num_subjects = norm_data.shape[0]
    num_rows = math.ceil(num_subjects / cols_per_row)
    
    # Initialize multi-plot figure
    fig, axes = plt.subplots(num_rows,
                             cols_per_row,
                             figsize = (cols_per_row * 3, num_rows * 2.5),
                             sharex = True,
                             sharey = True
                            )
    axes = axes.flatten()
    
    # Set main title
    fig.suptitle(f"Individual Plots\n{info['name']}", fontsize = 16, fontweight = 'bold', y = 1.02)
    
    # Iterate through subjects
    for i in range(num_subjects):
        ax = axes[i]
        
        # Draw individual plot
        ax.plot(endowment, norm_data[i, :], color = 'black', marker = '.', markersize = 4, linewidth = 1)
        
        # Add reference lines
        ax.axvline(80, color = 'gray',  linestyle = ':', alpha = 0.5)
        ax.axhline( 0, color = 'black', linewidth = 0.5, alpha = 0.3)
        
        # Format subplot
        ax.set_title(f"ID: {df_group.index[i]}", fontsize = 8)
        ax.set_ylim(-1.1, 1.1)
        ax.tick_params(axis = 'both', which = 'major', labelsize = 7)
    
    # Hide empty subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    # Export graph
    plt.tight_layout()
    plt.savefig(f'need_fulfilment_study_1_individual_plots_{g}.pdf', format = 'pdf', bbox_inches = 'tight')
    # plt.savefig(f'need_fulfilment_study_2_individual_plots_{g}.pdf', format = 'pdf', bbox_inches = 'tight')
    
    # Display graph
    plt.show()

## Perform t-Tests

In [ ]:
# Define function for t-Tests
def compare_groups(g1_num, g1_cols, g1_name, g2_num, g2_cols, g2_name):
    
    # Initialize list
    results = []
    
    # Define function for normalization
    def normalize_group(group_num, cols):
        
        # Filter data
        data = df_clean[df_clean['randnumber'] == group_num][cols].copy()
        
        # Iterate through cases
        for col in cols:
            
            # Convert to numeric
            data[col] = pd.to_numeric(data[col], errors = 'coerce')
        
        # Extract raw values to numpy array (Fix: added this line)
        raw_data = data.values
        
        # Calculate row-wise minimum and maximum
        with np.errstate(all = 'ignore'):
            row_min = np.nanmin(raw_data, axis = 1, keepdims = True)
            row_max = np.nanmax(raw_data, axis = 1, keepdims = True)
        
        # Calculate range and handle zero-variance cases to avoid division by zero
        diff = row_max - row_min
        diff[diff == 0] = np.nan
        
        # Apply subject-level normalization
        norm_data = 2 * (raw_data - row_min) / diff - 1
        return pd.DataFrame(norm_data, columns = cols)
    
    # Normalize groups
    norm_g1 = normalize_group(g1_num, g1_cols)
    norm_g2 = normalize_group(g2_num, g2_cols)
    
    # Iterate through cases
    for i in range(9):
        
        # Extract values
        vals_1 = norm_g1[g1_cols[i]].dropna().values
        vals_2 = norm_g2[g2_cols[i]].dropna().values
        
        # Run Welch's t-Test
        t_stat, p_val = ttest_ind(vals_1, vals_2, equal_var = False)
        
        # Determine sample sizes
        n1, n2 = len(vals_1), len(vals_2)
        
        # Calculate Cohen's d
        if n1 > 1 and n2 > 1:
            m1, m2 = np.mean(vals_1), np.mean(vals_2)
            v1, v2 = np.var(vals_1, ddof = 1), np.var(vals_2, ddof = 1)
            
            # Pooled standard deviation formula
            pooled_std = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
            cohens_d = (m1 - m2) / pooled_std
        else:
            m1, m2, cohens_d = np.nan, np.nan, np.nan
        
        # Store results
        results.append({
            'Endowment': endowment[i],
            f'Mean {g1_name}': round(m1, 3),
            f'Mean {g2_name}': round(m2, 3),
            't-Value': round(t_stat, 3),
            'p-Value': round(p_val, 3),
            'Significant': 'Yes' if p_val < 0.05 else '–',
            "Cohen's d": round(cohens_d, 3)
        })
    
    # Return results
    return pd.DataFrame(results)

# Execute
for gA_num, gB_num in comparisons:
    
    # Retrieve info for both groups
    infoA = group_infos[gA_num]
    infoB = group_infos[gB_num]
    
    # Generate results
    df_results = compare_groups(
        gA_num, infoA['cols'], infoA['name'],
        gB_num, infoB['cols'], infoB['name']
    )
    
    # Display results
    print(f"\n{infoA['name']} vs. {infoB['name']}")
    display(df_results)